### Imports and Loading Model

In [1]:
from sentence_transformers import SentenceTransformer, CrossEncoder, util
import chromadb
import numpy as np

# Setup
bi_model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
cross_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device="cpu")

### Indexing Data

In [2]:
documents = [
    "The Eiffel Tower is in Paris.",
    "The Louvre Museum is in Paris.",
    "Notre Dame Cathedral is in Paris.",
    "Montparnasse Tower is in Paris.",
    "The Statue of Liberty is in New York.",
    "The Colosseum is in Rome.",
    "Tokyo Tower is in Tokyo.",
    "The Burj Khalifa is in Dubai."
]

# Setup Chroma client
chroma_client = chromadb.Client()
# Drop existing collection if exists
try:
    chroma_client.delete_collection("demo")
except:
    pass
collection = chroma_client.create_collection("demo")

# Add documents with embeddings
embeddings = bi_model.encode(documents, device="cpu").tolist()
collection.add(documents=documents, embeddings=embeddings, ids=[str(i) for i in range(len(documents))])

### Baseline retrieval (bi-encoder)

In [3]:
query = "Where is the Eiffel Tower located?"
query_emb = bi_model.encode([query], device="cpu").tolist()
baseline_results = collection.query(query_embeddings=query_emb, n_results=3)["documents"][0]

print("Baseline Retrieval Results:")
for doc in baseline_results:
    print("-", doc)

Baseline Retrieval Results:
- The Eiffel Tower is in Paris.
- Montparnasse Tower is in Paris.
- Tokyo Tower is in Tokyo.


### Cross-encoder reranking

In [4]:
# Prepare (query, candidate) pairs
pairs = [(query, doc) for doc in baseline_results]
scores = cross_model.predict(pairs)

# Sort by cross-encoder score descending
reranked = [doc for _, doc in sorted(zip(scores, baseline_results), key=lambda x: x[0], reverse=True)]

print("\nReranked Results (Cross-Encoder):")
for doc, score in zip(reranked, sorted(scores, reverse=True)):
    print(f"- {doc} (score={score:.4f})")


Reranked Results (Cross-Encoder):
- The Eiffel Tower is in Paris. (score=9.3069)
- Montparnasse Tower is in Paris. (score=-0.3279)
- Tokyo Tower is in Tokyo. (score=-4.3100)


### Compare precision/accuracy

In [5]:
# For demo, assume correct answer contains "Eiffel Tower"
def precision_at_1(results):
    return int(any("Eiffel Tower" in doc for doc in results[:1]))

print("\nPrecision@1 Baseline:", precision_at_1(baseline_results))
print("Precision@1 Reranked:", precision_at_1(reranked))


Precision@1 Baseline: 1
Precision@1 Reranked: 1
